# 11주차 Type III 카이제곱 분석 Colab 실습노트

이 실습노트는 **`channel_purchase.csv`** 와 **`gender_preference.csv`** 를 활용하여,

- 카이제곱 **독립성 검정**
- 카이제곱 **동질성 검정**

을 Google Colab에서 직접 실행해 볼 수 있도록 구성한 자료입니다.

---

## 실습 목표

1. 범주형 데이터 문제에서 **교차표**를 만들 수 있다.
2. 카이제곱 검정의 **귀무가설/대립가설**을 설정할 수 있다.
3. `pd.crosstab()` 과 `chi2_contingency()` 를 사용해 Python으로 분석할 수 있다.
4. **p-value, 기대도수, 검정통계량**을 보고 결과를 해석할 수 있다.
5. 분석 결과를 **보고서 문장**으로 정리할 수 있다.

---

## 사용 파일

- `channel_purchase.csv`
- `gender_preference.csv`

학생들은 이 두 파일을 자신의 Google Drive 폴더에 올려 둔 뒤, 아래 실습을 순서대로 실행하면 됩니다.

## 0. Colab 시작 설정

아래 셀을 먼저 실행하세요.

- Google Drive 마운트
- CSV 폴더 경로 설정
- 파일 존재 여부 확인

In [ ]:
# 0-1. 구글 드라이브 마운트
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# 드라이브 연결
drive.mount('/content/drive')

# CSV가 저장된 폴더 경로 설정
# 예시 폴더: MyDrive/type3_week11
DATA_DIR = Path('/content/drive/MyDrive/type3_week11')

print('설정된 데이터 폴더:', DATA_DIR)

In [ ]:
# 0-2. 파일 존재 여부 확인
required_files = ['channel_purchase.csv', 'gender_preference.csv']

print('[파일 존재 여부 확인]')
for fname in required_files:
    fpath = DATA_DIR / fname
    print(f'{fname}:', 'OK' if fpath.exists() else '없음')

---
# 실습 1. channel_purchase.csv : 카이제곱 독립성 검정

## 1-1. 문제 제시

한 온라인 쇼핑몰은 고객의 **유입채널(`channel`)** 과 **구매여부(`purchase_yn`)** 사이에 관련성이 있는지 알고 싶어 한다.

유입채널은 다음과 같이 구성되어 있다.
- 검색광고
- SNS
- 이메일
- 직접방문

구매여부는 다음과 같이 구성되어 있다.
- 1: 구매
- 0: 비구매

이때, 유입채널과 구매여부가 서로 독립인지 카이제곱 검정을 통해 확인하시오.

---

## 1-2. 가설 설정

- **귀무가설(H0)**: 유입채널과 구매여부는 서로 독립이다.
- **대립가설(H1)**: 유입채널과 구매여부는 서로 독립이 아니다.

---

## 1-3. 해석 포인트

- `pd.crosstab()` 으로 **교차표**를 만든다.
- `chi2_contingency()` 로 **카이제곱 통계량, p-value, 자유도, 기대도수**를 구한다.
- `p-value < 0.05` 이면 유입채널과 구매여부는 **관련이 있다**고 해석한다.

In [ ]:
# 1-4. 데이터 불러오기
file_path = DATA_DIR / 'channel_purchase.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('
[기본 정보]')
print(df.info())

In [ ]:
# 1-5. 교차표 생성
ct = pd.crosstab(df['channel'], df['purchase_yn'])

print('[교차표]')
print(ct)

In [ ]:
# 1-6. 카이제곱 독립성 검정
chi2, p, dof, expected = chi2_contingency(ct)
expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

print('
[기대도수]')
print(expected_df)

In [ ]:
# 1-7. 채널별 구매전환율 확인
conversion_rate = pd.crosstab(df['channel'], df['purchase_yn'], normalize='index') * 100

print('[채널별 구매 비율(%)]')
print(conversion_rate.round(2))

print('
[최종 해석 출력]')
if p < 0.05:
    print('p-value가 0.05보다 작으므로, 유입채널과 구매여부는 독립이 아니며 서로 관련이 있다고 해석할 수 있습니다.')
else:
    print('p-value가 0.05 이상이므로, 유입채널과 구매여부의 관련성을 확인하기 어렵습니다.')

best_channel = conversion_rate[1].idxmax() if 1 in conversion_rate.columns else conversion_rate.iloc[:, -1].idxmax()
best_rate = conversion_rate[1].max() if 1 in conversion_rate.columns else conversion_rate.iloc[:, -1].max()
print(f'구매 비율이 가장 높은 채널은 {best_channel}이며, 구매 비율은 {best_rate:.2f}%입니다.')

## 1-8. 결과 해석 예시

카이제곱 독립성 검정 결과 p-value가 유의수준 0.05보다 작으면, **유입채널과 구매여부는 서로 독립이 아니며 통계적으로 유의한 관련성이 있다**고 해석한다.

또한 채널별 구매 비율을 함께 보면, 어떤 채널이 실제로 더 높은 전환 성과를 보이는지 추가로 설명할 수 있다.

## 1-9. 보고서 작성 예시

> `channel_purchase.csv` 데이터를 이용하여 유입채널과 구매여부의 독립성 여부를 검정하였다. `pd.crosstab()`으로 교차표를 생성한 뒤 `chi2_contingency()`를 적용한 결과, p-value가 0.05보다 작게 나타났다. 따라서 유입채널과 구매여부는 서로 독립이 아니며, 통계적으로 유의한 관련성이 있다고 해석할 수 있다. 또한 채널별 구매 비율을 비교한 결과, 가장 높은 전환율을 보인 채널과 가장 낮은 전환율을 보인 채널 사이에 차이가 확인되었다.`

---
# 실습 2. gender_preference.csv : 카이제곱 동질성 검정

## 2-1. 문제 제시

한 교육 플랫폼은 **성별(`gender`)** 에 따라 **콘텐츠 선호(`content_type`) 분포**가 같은지 알고 싶어 한다.

콘텐츠 유형은 다음과 같이 구성되어 있다.
- 데이터분석
- 마케팅
- 디자인
- 프로그래밍

이때, 남성과 여성의 콘텐츠 선호 분포가 같은지 카이제곱 검정을 통해 확인하시오.

---

## 2-2. 가설 설정

- **귀무가설(H0)**: 성별에 따른 콘텐츠 선호 분포는 같다.
- **대립가설(H1)**: 성별에 따른 콘텐츠 선호 분포는 다르다.

---

## 2-3. 해석 포인트

- 동질성 검정은 **집단 간 분포가 같은지**를 확인하는 문제이다.
- 계산 함수는 독립성 검정과 같지만, 해석 문장은 **분포가 동일/상이하다**로 표현한다.
- 성별별 최다 선호 콘텐츠도 함께 확인하면 보고서가 더 풍부해진다.

In [ ]:
# 2-4. 데이터 불러오기
file_path = DATA_DIR / 'gender_preference.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('
[기본 정보]')
print(df.info())

In [ ]:
# 2-5. 교차표 생성
ct = pd.crosstab(df['gender'], df['content_type'])

print('[교차표]')
print(ct)

In [ ]:
# 2-6. 카이제곱 동질성 검정
chi2, p, dof, expected = chi2_contingency(ct)
expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

print('
[기대도수]')
print(expected_df)

In [ ]:
# 2-7. 성별 내 비율과 최다 선호 콘텐츠 확인
ratio = pd.crosstab(df['gender'], df['content_type'], normalize='index') * 100
top_pref = ct.idxmax(axis=1)

print('[성별 내 콘텐츠 선호 비율(%)]')
print(ratio.round(2))

print('
[성별별 최다 선호 콘텐츠]')
print(top_pref)

print('
[최종 해석 출력]')
if p < 0.05:
    print('p-value가 0.05보다 작으므로, 성별에 따라 콘텐츠 선호 분포가 다르다고 해석할 수 있습니다.')
else:
    print('p-value가 0.05 이상이므로, 성별에 따른 콘텐츠 선호 분포 차이를 확인하기 어렵습니다.')

for g in top_pref.index:
    print(f'{g}의 최다 선호 콘텐츠는 {top_pref[g]}입니다.')

## 2-8. 결과 해석 예시

카이제곱 동질성 검정 결과 p-value가 유의수준 0.05보다 작으면, **남성과 여성의 콘텐츠 선호 분포는 동일하지 않다**고 해석한다.

이때 단순히 p-value만 제시하지 말고, 성별별 최다 선호 콘텐츠나 행 비율표를 함께 활용하면 결과 설명이 더 명확해진다.

## 2-9. 보고서 작성 예시

> `gender_preference.csv` 데이터를 이용하여 성별에 따른 콘텐츠 선호 분포가 동일한지 검정하였다. `pd.crosstab()`으로 성별과 콘텐츠 유형의 교차표를 만든 뒤 `chi2_contingency()`를 적용한 결과, p-value가 0.05보다 작게 나타났다. 따라서 성별에 따른 콘텐츠 선호 분포는 동일하지 않으며, 성별에 따라 선호 유형 차이가 존재한다고 해석할 수 있다. 추가로 성별별 비율을 확인한 결과, 각 집단에서 상대적으로 선호하는 콘텐츠 유형이 다르게 나타났다.`

---
# 정리

## 오늘 실습에서 기억해야 할 것

### 1. 독립성 검정
- 한 데이터 안에서 **두 범주형 변수의 관련성**을 확인
- 예: 유입채널과 구매여부

### 2. 동질성 검정
- 여러 집단에서 **범주형 분포가 같은지** 확인
- 예: 성별에 따른 콘텐츠 선호 분포

### 3. 핵심 Python method
- `pd.read_csv()`
- `pd.crosstab()`
- `chi2_contingency()`

### 4. 해석 기준
- `p-value < 0.05` 이면 귀무가설 기각
- 기대도수와 실제도수 차이가 클수록 카이제곱 통계량이 커질 수 있음

---

## 과제 제안

1. 두 CSV에 대해 직접 교차표를 다시 작성해 보기
2. p-value를 기준으로 해석 문장을 스스로 써 보기
3. 비율표를 이용해 어떤 범주가 가장 큰 차이를 보이는지 설명해 보기